# Phishing Detection - Data Science in Cyber Final Project

This notebook reproduces and critically evaluates the selected source: **Phishing Website Detection by Machine Learning Techniques** by `shreyagopal`.

The analysis follows concepts covered in the course lectures: data loading, EDA, missing/special values, correlation and association, train/test methodology, feature engineering, model comparison, evaluation metrics, error analysis, and cybersecurity interpretation of false positives and false negatives.

## 1. Imports and configuration

The notebook uses fixed random seeds and writes generated artifacts to `figures/` and `outputs/`.

In [ ]:
from pathlib import Path
import json
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split, GroupShuffleSplit, StratifiedKFold, cross_validate
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score, fbeta_score,
    matthews_corrcoef, roc_auc_score, confusion_matrix
)
from sklearn.inspection import permutation_importance

warnings.filterwarnings('ignore')
RANDOM_STATE = 42
BASE = Path.cwd()
DATA_DIR = BASE / 'data'
FIG_DIR = BASE / 'figures'
OUT_DIR = BASE / 'outputs'
for path in [DATA_DIR, FIG_DIR, OUT_DIR]:
    path.mkdir(parents=True, exist_ok=True)

pd.set_option('display.max_columns', 100)
pd.set_option('display.width', 160)

## 2. Data loading and inspection

The source repository provides a final extracted dataset, `5.urldata.csv`, containing 10,000 URL/domain records and 17 URL/domain/HTML/JavaScript features. The label is binary: `1` for phishing and `0` for legitimate.

In [ ]:
df = pd.read_csv(DATA_DIR / '5.urldata.csv')
df = df.rename(columns={c: c.replace('/', '_').replace(' ', '_') for c in df.columns})
feature_cols_original = [c for c in df.columns if c not in ['Domain', 'Label']]

print('Dataset shape:', df.shape)
display(df.head())

inspection_table = pd.DataFrame({
    'dtype': df.dtypes.astype(str),
    'missing_count': df.isna().sum(),
    'unique_values': df.nunique(dropna=False),
    'example_value': df.iloc[0].astype(str)
})
inspection_table.to_csv(OUT_DIR / 'data_inspection.csv')
display(inspection_table)

inspection = {
    'index_is_unique': bool(df.index.is_unique),
    'has_temporal_column': any(('date' in c.lower() or 'time' in c.lower() or 'timestamp' in c.lower()) for c in df.columns),
    'duplicated_rows': int(df.duplicated().sum()),
    'duplicated_domains': int(df['Domain'].duplicated().sum()),
    'unique_domains': int(df['Domain'].nunique()),
    'single_value_columns': [c for c in df.columns if df[c].nunique(dropna=False) <= 1]
}
inspection

### Interpretation

The dataset has no missing values and no temporal column. The absence of timestamp information is important: phishing campaigns evolve quickly, so temporal validation would be preferable in a production setting. The dataset also contains many duplicate rows/domains, so a random split may overestimate performance.

## 3. Class prevalence and exploratory data analysis

The dataset is balanced, but a real-world stream of URLs is not expected to contain 50% phishing. Therefore, accuracy is not enough; precision, recall, F1/F2, MCC, ROC-AUC, and confusion-matrix analysis are reported later.

In [ ]:
class_counts = df['Label'].value_counts().sort_index()
class_table = pd.DataFrame({
    'count': class_counts,
    'percent': (df['Label'].value_counts(normalize=True).sort_index() * 100).round(2)
})
class_table.index = ['Legitimate (0)', 'Phishing (1)']
display(class_table)

plt.figure(figsize=(5, 4))
class_counts.plot(kind='bar')
plt.xticks([0, 1], ['Legitimate (0)', 'Phishing (1)'], rotation=0)
plt.ylabel('Count')
plt.title('Class Distribution')
plt.tight_layout()
plt.savefig(FIG_DIR / 'class_distribution.png', dpi=180)
plt.show()

plt.figure(figsize=(7, 4))
for label, label_name in [(0, 'Legitimate'), (1, 'Phishing')]:
    df.loc[df['Label'] == label, 'URL_Depth'].plot(kind='hist', bins=18, alpha=0.55, label=label_name)
plt.title('URL Depth Distribution by Class')
plt.xlabel('URL Depth')
plt.legend()
plt.tight_layout()
plt.savefig(FIG_DIR / 'url_depth_distribution.png', dpi=180)
plt.show()

### Crosstabs and class-conditional feature prevalence

Crosstabs are suitable for the binary/ordinal cybersecurity indicators. They show how indicators such as `Web_Traffic`, `Domain_Age`, `DNS_Record`, `Prefix_Suffix`, and `TinyURL` differ between phishing and legitimate URLs.

In [ ]:
binary_cols = [c for c in feature_cols_original if df[c].nunique() == 2]
mean_by_class = df.groupby('Label')[binary_cols].mean().T
mean_by_class['abs_gap'] = (mean_by_class[1] - mean_by_class[0]).abs()
mean_top = mean_by_class.sort_values('abs_gap', ascending=False).head(12)
mean_top.to_csv(OUT_DIR / 'binary_prevalence_by_class.csv')
display(mean_top)

mean_top[[0, 1]].plot(kind='barh', figsize=(8, 6))
plt.title('Top Binary Feature Prevalence by Class')
plt.xlabel('Feature mean / prevalence')
plt.tight_layout()
plt.savefig(FIG_DIR / 'binary_feature_prevalence.png', dpi=180)
plt.show()

for col in ['Web_Traffic', 'Domain_Age', 'DNS_Record', 'Prefix_Suffix', 'TinyURL']:
    print('
Crosstab for', col)
    ctab = pd.crosstab(df[col], df['Label'], normalize='index').rename(columns={0: 'Legitimate_rate', 1: 'Phishing_rate'})
    ctab.to_csv(OUT_DIR / f'crosstab_{col}.csv')
    display(ctab.round(3))

group_summary = df.groupby('Label')[['URL_Depth', 'URL_Length', 'Redirection', 'Web_Forwards']].agg(['mean', 'median', 'std']).round(3)
group_summary.to_csv(OUT_DIR / 'group_summary.csv')
display(group_summary)

## 4. Correlation and redundancy analysis

Spearman correlation is used because most variables are binary or ordinal and not continuous normal variables. Spearman measures monotonic rank association and is more suitable here than Pearson. Kendall would also be possible, but Spearman is sufficiently stable for this dataset size.

In [ ]:
spearman = df[feature_cols_original + ['Label']].corr(method='spearman')
top_corr = spearman['Label'].drop('Label').abs().sort_values(ascending=False).head(12)
top_corr.to_csv(OUT_DIR / 'top_spearman_correlations.csv')
display(top_corr.to_frame('|Spearman rho with Label|').round(4))

top_corr.sort_values().plot(kind='barh', figsize=(8, 5))
plt.title('Top Absolute Spearman Correlations with Label')
plt.xlabel('|Spearman rho|')
plt.tight_layout()
plt.savefig(FIG_DIR / 'top_spearman_correlations.png', dpi=180)
plt.show()

corr_abs = spearman.drop(index='Label', columns='Label').abs()
pairs = []
cols = list(corr_abs.columns)
for i in range(len(cols)):
    for j in range(i + 1, len(cols)):
        value = float(corr_abs.iloc[i, j])
        if value >= 0.70:
            pairs.append({'feature_1': cols[i], 'feature_2': cols[j], 'abs_spearman': value})
redundant_pairs = pd.DataFrame(pairs).sort_values('abs_spearman', ascending=False) if pairs else pd.DataFrame(columns=['feature_1', 'feature_2', 'abs_spearman'])
redundant_pairs.to_csv(OUT_DIR / 'high_correlation_pairs.csv', index=False)
display(redundant_pairs)

## 5. Feature engineering

The original source already extracted 17 features. I add domain-structure features from the `Domain` string. I do **not** one-hot encode the raw domain name because thousands of domain categories would encourage memorization rather than general phishing detection.

In [ ]:
def extract_domain_features(domain: str) -> pd.Series:
    s = str(domain).strip().lower()
    parts = s.split('.')
    tld = parts[-1] if len(parts) > 1 else ''
    common_tlds = {'com', 'org', 'net', 'edu', 'gov', 'co', 'io', 'jp', 'uk', 'de', 'fr', 'ru', 'br', 'in', 'au', 'ca'}
    return pd.Series({
        'domain_length': len(s),
        'domain_digit_count': sum(ch.isdigit() for ch in s),
        'domain_hyphen_count': s.count('-'),
        'domain_dot_count': s.count('.'),
        'subdomain_count': max(len(parts) - 2, 0),
        'has_www': int(s.startswith('www.')),
        'tld_length': len(tld),
        'is_common_tld': int(tld in common_tlds),
    })

engineered = df['Domain'].apply(extract_domain_features)
df_fe = pd.concat([df, engineered], axis=1)
engineered_cols = list(engineered.columns)
feature_cols = feature_cols_original + engineered_cols

df_fe[['Domain'] + feature_cols + ['Label']].to_csv(DATA_DIR / 'phishing_processed_features.csv', index=False)
print('Engineered features:', engineered_cols)
display(df_fe.groupby('Label')[engineered_cols].mean().round(3))

## 6. Model training

The notebook trains four models: Logistic Regression, Decision Tree, Random Forest, and Gradient Boosting. Logistic Regression is scaled; tree-based models do not require scaling. Two split strategies are used:

1. Random stratified 80/20 split, similar to the selected source.
2. Grouped split by domain, a stricter robustness check against duplicated or related domains.

In [ ]:
X = df_fe[feature_cols]
y = df_fe['Label']
groups = df_fe['Domain']

X_train, X_test, y_train, y_test, domain_train, domain_test = train_test_split(
    X, y, groups, test_size=0.20, stratify=y, random_state=RANDOM_STATE
)

gss = GroupShuffleSplit(n_splits=1, test_size=0.20, random_state=RANDOM_STATE)
g_train_idx, g_test_idx = next(gss.split(X, y, groups=groups))
Xg_train, Xg_test = X.iloc[g_train_idx], X.iloc[g_test_idx]
yg_train, yg_test = y.iloc[g_train_idx], y.iloc[g_test_idx]

models = {
    'Logistic Regression': Pipeline([
        ('scaler', StandardScaler()),
        ('model', LogisticRegression(max_iter=2000, class_weight='balanced', random_state=RANDOM_STATE))
    ]),
    'Decision Tree': DecisionTreeClassifier(max_depth=8, min_samples_leaf=20, random_state=RANDOM_STATE, class_weight='balanced'),
    'Random Forest': RandomForestClassifier(n_estimators=120, min_samples_leaf=2, random_state=RANDOM_STATE, class_weight='balanced', n_jobs=1),
    'Gradient Boosting': GradientBoostingClassifier(random_state=RANDOM_STATE)
}

print('Random split:', len(y_train), 'train /', len(y_test), 'test')
print('Grouped split:', len(yg_train), 'train /', len(yg_test), 'test')
print('Grouped test prevalence:')
display((yg_test.value_counts(normalize=True).sort_index() * 100).round(2))

## 7. Evaluation metrics

For phishing detection, false negatives expose users to phishing pages. False positives block legitimate activity and create alert fatigue. Therefore the evaluation reports Accuracy, Precision, Recall, F1, F2, MCC, ROC-AUC, and confusion-matrix counts.

In [ ]:
def evaluate_model(model, Xtr, Xte, ytr, yte):
    model.fit(Xtr, ytr)
    pred = model.predict(Xte)
    scores = model.predict_proba(Xte)[:, 1] if hasattr(model, 'predict_proba') else pred
    tn, fp, fn, tp = confusion_matrix(yte, pred).ravel()
    metrics = {
        'Accuracy': accuracy_score(yte, pred),
        'Precision': precision_score(yte, pred, zero_division=0),
        'Recall': recall_score(yte, pred, zero_division=0),
        'F1': f1_score(yte, pred, zero_division=0),
        'F2': fbeta_score(yte, pred, beta=2, zero_division=0),
        'MCC': matthews_corrcoef(yte, pred),
        'ROC_AUC': roc_auc_score(yte, scores),
        'TN': int(tn), 'FP': int(fp), 'FN': int(fn), 'TP': int(tp)
    }
    return metrics, model, pred, scores

random_rows, group_rows = [], []
fitted_models, prediction_cache = {}, {}
for name, model in models.items():
    metrics, fitted, pred, scores = evaluate_model(model, X_train, X_test, y_train, y_test)
    metrics['Model'] = name
    random_rows.append(metrics)
    fitted_models[name] = fitted
    prediction_cache[name] = (pred, scores)

    group_metrics, _, _, _ = evaluate_model(model, Xg_train, Xg_test, yg_train, yg_test)
    group_metrics['Model'] = name
    group_rows.append(group_metrics)

random_results = pd.DataFrame(random_rows).set_index('Model').sort_values('F1', ascending=False)
group_results = pd.DataFrame(group_rows).set_index('Model').sort_values('F1', ascending=False)
random_results.to_csv(OUT_DIR / 'model_results_random_split.csv')
group_results.to_csv(OUT_DIR / 'model_results_group_split.csv')

display(random_results.round(4))
display(group_results.round(4))

## 8. Cross-validation, explainability, and error analysis

In [ ]:
# Cross-validation.
scoring = {'accuracy': 'accuracy', 'precision': 'precision', 'recall': 'recall', 'f1': 'f1', 'roc_auc': 'roc_auc'}
cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=RANDOM_STATE)
cv_rows = []
for name, model in models.items():
    scores = cross_validate(model, X, y, cv=cv, scoring=scoring, n_jobs=1)
    row = {'Model': name}
    for metric_name in scoring:
        values = scores[f'test_{metric_name}']
        row[f'{metric_name}_mean'] = np.mean(values)
        row[f'{metric_name}_std'] = np.std(values)
    cv_rows.append(row)
cv_results = pd.DataFrame(cv_rows).set_index('Model').sort_values('f1_mean', ascending=False)
cv_results.to_csv(OUT_DIR / 'cross_validation_results.csv')
display(cv_results.round(4))

# Model comparison plot.
random_results[['Accuracy', 'Precision', 'Recall', 'F1', 'ROC_AUC']].sort_values('F1').plot(kind='barh', figsize=(8, 5))
plt.title('Model Comparison - Random Stratified Test Split')
plt.xlabel('Metric value')
plt.xlim(0, 1)
plt.tight_layout()
plt.savefig(FIG_DIR / 'model_comparison.png', dpi=180)
plt.show()

best_name = random_results['F1'].idxmax()
best_model = fitted_models[best_name]
best_pred, best_scores = prediction_cache[best_name]

cm = confusion_matrix(y_test, best_pred)
plt.figure(figsize=(5, 4))
plt.imshow(cm)
plt.title(f'Confusion Matrix - {best_name}')
plt.xlabel('Predicted label')
plt.ylabel('True label')
plt.xticks([0, 1], ['Legitimate', 'Phishing'])
plt.yticks([0, 1], ['Legitimate', 'Phishing'])
for (i, j), value in np.ndenumerate(cm):
    plt.text(j, i, str(value), ha='center', va='center')
plt.tight_layout()
plt.savefig(FIG_DIR / 'confusion_matrix_best.png', dpi=180)
plt.show()

# Feature importance.
if hasattr(best_model, 'feature_importances_'):
    importances = pd.Series(best_model.feature_importances_, index=feature_cols).sort_values(ascending=False)
else:
    perm = permutation_importance(best_model, X_test, y_test, n_repeats=3, random_state=RANDOM_STATE, n_jobs=1)
    importances = pd.Series(perm.importances_mean, index=feature_cols).sort_values(ascending=False)
importances.to_csv(OUT_DIR / 'feature_importance_best_model.csv')
display(importances.head(15).to_frame('importance').round(4))

importances.head(15).sort_values().plot(kind='barh', figsize=(8, 5))
plt.title(f'Top Feature Importances - {best_name}')
plt.xlabel('Importance')
plt.tight_layout()
plt.savefig(FIG_DIR / 'feature_importance.png', dpi=180)
plt.show()

## 9. Error analysis and threshold trade-off

The threshold table shows the practical security trade-off: reducing the threshold improves recall but increases false positives.

In [ ]:
errors = df_fe.loc[X_test.index, ['Domain', 'Label'] + feature_cols].copy()
errors['predicted'] = best_pred
errors['phishing_score'] = best_scores
errors_only = errors[errors['Label'] != errors['predicted']]

false_negatives = errors_only[(errors_only['Label'] == 1) & (errors_only['predicted'] == 0)].sort_values('phishing_score').head(10)
false_positives = errors_only[(errors_only['Label'] == 0) & (errors_only['predicted'] == 1)].sort_values('phishing_score', ascending=False).head(10)
false_negatives.to_csv(OUT_DIR / 'false_negatives_examples.csv', index=False)
false_positives.to_csv(OUT_DIR / 'false_positives_examples.csv', index=False)

print('False negatives - phishing predicted as legitimate')
display(false_negatives[['Domain', 'Label', 'predicted', 'phishing_score'] + feature_cols_original[:6]])
print('False positives - legitimate predicted as phishing')
display(false_positives[['Domain', 'Label', 'predicted', 'phishing_score'] + feature_cols_original[:6]])

threshold_rows = []
for threshold in [0.30, 0.35, 0.40, 0.45, 0.50, 0.60, 0.70]:
    threshold_pred = (best_scores >= threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_test, threshold_pred).ravel()
    threshold_rows.append({
        'threshold': threshold,
        'precision': precision_score(y_test, threshold_pred, zero_division=0),
        'recall': recall_score(y_test, threshold_pred, zero_division=0),
        'f1': f1_score(y_test, threshold_pred, zero_division=0),
        'f2': fbeta_score(y_test, threshold_pred, beta=2, zero_division=0),
        'FP': int(fp), 'FN': int(fn), 'TP': int(tp), 'TN': int(tn)
    })
threshold_df = pd.DataFrame(threshold_rows)
threshold_df.to_csv(OUT_DIR / 'threshold_tradeoff.csv', index=False)
display(threshold_df.round(4))

## 10. Executive summary / summing it up

The source is reproducible at the dataset and modeling level because it provides the extracted dataset and model notebooks. It is weaker at the data-generation level because phishing pages change quickly and the final table lacks timestamps and sampling metadata.

The reproduction supports the general claim that URL/domain/HTML/JavaScript features can detect phishing on this dataset. However, the dataset has many duplicate rows and domains, and the grouped split produces lower performance than the random split. Therefore, the source's conclusions are directionally correct but not sufficient as production evidence.

For deployment, the model should be evaluated on newer URLs using temporal splits, explicit prevalence assumptions, and threshold tuning based on the cost of false negatives and false positives.

In [ ]:
summary = {
    'shape': list(df.shape),
    'class_counts': class_counts.to_dict(),
    'missing_values_total': int(df.isna().sum().sum()),
    'duplicated_rows': int(df.duplicated().sum()),
    'duplicated_domains': int(df['Domain'].duplicated().sum()),
    'unique_domains': int(df['Domain'].nunique()),
    'has_temporal_column': inspection['has_temporal_column'],
    'top_spearman_abs': top_corr.round(4).to_dict(),
    'engineered_features': engineered_cols,
    'best_model_random_split': best_name,
    'best_model_metrics_random_split': random_results.loc[best_name].round(4).to_dict(),
    'best_group_split_model_by_f1': group_results['F1'].idxmax(),
    'best_group_split_metrics': group_results.iloc[0].round(4).to_dict(),
    'top_feature_importances': importances.head(10).round(4).to_dict(),
}
with open(OUT_DIR / 'analysis_summary.json', 'w', encoding='utf-8') as f:
    json.dump(summary, f, indent=2)
print(json.dumps(summary, indent=2))